# N033 · 二维前缀和与容斥

**目标：** 用四项容斥查询矩形和且统一坐标约定。

**先修：** N018, N031。本章自包含，依次运行即可；建议先读推导，再预测输出。

**知识点：** 二维累加；边界补零；半开矩形；容斥。

**配套讲解来源：** [同名逐章意见](../../comment/033_prefix_sum_2d.md)。

本章阅读顺序：**先明白问题 → 补基础 → 手算 → 看核心片段 → 分段运行 → 核对测试**。需要复制使用时，跳到末尾“完整代码”，它包含全部依赖、实现和测试。

## 1. 通俗解释：输入什么，要得到什么

这一章把 N031 的一维前缀和搬到二维：给你一个**固定不变**的矩阵，之后会问很多次"某个矩形区域里所有数的和是多少"，我们希望每次查询都 O(1) 出答案。

具体例子：矩阵

```
1 2
3 4
```

查询左上角 (r1=0,c1=0)、右下角 (r2=2,c2=2) 的矩形（半开约定，行取 [0,2)、列取 [0,2)，即整个矩阵），答案是 `10`，因为 1+2+3+4=10。查询 (1,1,2,2)（行 [1,2)、列 [1,2)，只有右下角一个格子），答案是 `4`。再查询 (0,1,2,2)（行 [0,2)、列 [1,2)，即右列 2 和 4），答案是 `6`。你会发现四个参数里"贴边"的查询（从 0 开始、到 rows/cols 结束）不需要任何特殊写法，这要归功于接下来讲的补零边界。

本章还带一个应用题**矩阵块和**：对矩阵每个位置 (i,j)，计算以它为中心、向外扩展 k 格的矩形范围（越界就贴边截断）内所有元素之和，输出一个同尺寸的答案矩阵。比如上例 k=3 时扩展范围盖住整个 2×2 矩阵，所以答案矩阵每个位置都是 10。

难点在两处：一是二维的前缀怎么定义和递推；二是查询矩形时，直接"大减大"会把一块区域多减一次，需要**容斥**补回来——这正是本章标题里的词。

## 2. 基础知识：先读懂这些词

- **二维前缀和 P**：P[r][c] 表示"前 r 行、前 c 列"这个左上角矩形里所有元素的和（行半开 [0,r)、列半开 [0,c)）。它是把一维前缀"从开头到 i"的概念在行、列两个方向同时做一遍。
- **边界补零**：给前缀表多开一行、多开一列并填 0——P 尺寸是 (rows+1)×(cols+1)，第 0 行和第 0 列全为 0。这样"贴着上边或左边的矩形"查询时公式不用变，省掉一堆 if 特判。这和一维里 P[0]=0 的道理一样。
  - 直观图景：想象把原矩阵往右下挪一格，左边和顶上各镶一圈 0。查询矩形 (0,0,2,2) 时"上方条"和"左方条"分别落在零圈上，减 0 等于没减，公式自然过渡到贴边情形。
  - 不补零行不行？行，但 query(0,…)、query(…,0) 全要写 `if` 分支或把下标平移一位，代码里多出的判断比省下的内存值钱得多。
- **容斥（inclusion–exclusion）**：统计多个互相重叠的集合时"多加的减掉、多减的加回"的校正方法。本章两处用到：建表时 `P[i+1][j+1] = mat[i][j] + P[i][j+1] + P[i+1][j] − P[i][j]`，查询时 `矩形和 = P[r2][c2] − P[r1][c2] − P[r2][c1] + P[r1][c1]`。两处的核心都是"左上角那块被算了两次，要减掉或加回一次"。生活化的类比：两个人合买水果各报一份清单算总价，苹果在两张清单里都有，直接相加会把苹果算两次钱，得减掉一份。
- **半开矩形**：本章 query 的四个参数 (r1,c1,r2,c2) 表示行 [r1,r2)、列 [c1,c2)，即包含 r1 行不包含 r2 行。和一维半开区间一样，它让"空矩形""整块""贴边"都套同一个公式。行数是 r2−r1、列数是 c2−c1，矩形内格子数就是两者相乘，不需要任何加一减一。
- **PrefixSum2D 类**：封装"O(RC) 建表 + O(1) 查询"的数据结构，与 N031 的 PrefixSum 类是同一个思路的二维版。
- **matrix_block_sum 的边界钳制（clamp）**：计算中心块范围时用 `max(0, i−k)` 和 `min(rows, i+k+1)` 把越界的坐标拉回合法范围。注意 `i+k+1` 的 +1 是半开右端；用 min/max 钳制后，贴边的中心也能直接复用同一个 query，不必写特例。

## 3. 思路推导：从小例子开始

问题主线（二维前缀和）：

- **Step 1**：定义 P[r][c] = 行 [0,r)、列 [0,c) 内所有元素之和，并给第 0 行、第 0 列补 0。
- **Step 2**：递推建表。要算 P[i+1][j+1]（即把 mat[i][j] 纳入左上矩形），朴素想法是"左边的前缀 + 上面的前缀 + 自己"。但左前缀 P[i+1][j] 和上前缀 P[i][j+1] 都各自包含了左上角那块 P[i][j]，加起来时左上角被算了两次，所以再减掉一个 P[i][j]。
  - 一维版本是"新前缀 = 旧前缀 + 自己"，二维多出一个方向就多出一层重叠，容斥项也从 0 个变 1 个。这个"加一个方向、补一次重叠"的规律可以推广到更高维。
- **Step 3**：查询任意半开矩形 (r1,c1,r2,c2)。用"大矩形减两条边"：P[r2][c2] 是从 (0,0) 到 (r2,c2) 的大块；减去目标上方一条 P[r1][c2] 和左边一条 P[r2][c1]；这两条在左上角 P[r1][c1] 处重叠被减了两次，加回一次。
- **Step 4**：matrix_block_sum 里对每个 (i,j) 把范围钳制成 (max(0,i−k), max(0,j−k), min(rows,i+k+1), min(cols,j+k+1))，直接调 query。

手算演示（矩阵 `[[1,2],[3,4]]`，“运行示例”部分 的数据）建表，P 尺寸 3×3，初始全 0：

- P[1][1] = mat[0][0] + P[0][1] + P[1][0] − P[0][0] = 1+0+0−0 = **1**
- P[1][2] = mat[0][1] + P[0][2] + P[1][1] − P[0][1] = 2+0+1−0 = **3**
- P[2][1] = mat[1][0] + P[1][1] + P[2][0] − P[1][0] = 3+1+0−0 = **4**
- P[2][2] = mat[1][1] + P[1][2] + P[2][1] − P[1][1] = 4+3+4−1 = **10**

得到前缀表（第 0 行第 0 列补零）：

```
0 0  0
0 1  3
0 4 10
```

这就是 notebook “运行示例”部分 打印的第一张表。用它查三个矩形（“运行示例”部分 第二张表的数据）：

- query(0,0,2,2) = P[2][2] − P[0][2] − P[2][0] + P[0][0] = 10−0−0+0 = **10**（整块）
- query(1,1,2,2) = P[2][2] − P[1][2] − P[2][1] + P[1][1] = 10−3−4+1 = **4**（右下单格）
- query(0,1,2,2) = P[2][2] − P[0][2] − P[2][1] + P[0][1] = 10−0−4+0 = **6**（右列 2+4）

你可以逐个对着矩阵手验：整块 10、右下角 4、右列 6，全部吻合。

再看 matrix_block_sum 的钳制怎么落在这张表上（k=3、矩阵 2×2）：对中心 (0,0)，行界是 `max(0, 0−3)=0` 到 `min(2, 0+4)=2`，列界同算，得到 (0,0,2,2)——四个方向的理想边界全部越界，全部被 max/min 拉回，最终每个中心都退化成整块查询，所以答案矩阵全是 10。`i+k+1` 里那个 +1 别漏：中心想覆盖到第 i+k **行**（含），而 query 的行右端是半开的，所以右端要写成 i+k+1。

## 4. 核心代码：PrefixSum2D

先关注决定答案的状态与更新操作。这是完整实现的对应片段，不是另一套解法；导入、辅助结构及其余接口在下一节补全。

```python
class PrefixSum2D:

    def __init__(self, mat):
        self.rows = len(mat)
        self.cols = len(mat[0]) if mat else 0
        self.p = [[0] * (self.cols + 1) for _ in range(self.rows + 1)]
        for i in range(self.rows):
            for j in range(self.cols):
                self.p[i + 1][j + 1] = mat[i][j] + self.p[i][j + 1] + self.p[i + 1][j] - self.p[i][j]

    def query(self, r1, c1, r2, c2):
        if not (0 <= r1 <= r2 <= self.rows and 0 <= c1 <= c2 <= self.cols):
            raise IndexError('invalid rectangle')
        p = self.p
        return p[r2][c2] - p[r1][c2] - p[r2][c1] + p[r1][c1]
```

### 逐段读懂代码

先看 `PrefixSum2D` 类，对应实现如下：

```python
class PrefixSum2D:

    def __init__(self, mat):
        self.rows = len(mat)
        self.cols = len(mat[0]) if mat else 0
        self.p = [[0] * (self.cols + 1) for _ in range(self.rows + 1)]
        for i in range(self.rows):
            for j in range(self.cols):
                self.p[i + 1][j + 1] = mat[i][j] + self.p[i][j + 1] + self.p[i + 1][j] - self.p[i][j]

    def query(self, r1, c1, r2, c2):
        if not (0 <= r1 <= r2 <= self.rows and 0 <= c1 <= c2 <= self.cols):
            raise IndexError('invalid rectangle')
        p = self.p
        return p[r2][c2] - p[r1][c2] - p[r2][c1] + p[r1][c1]
```

- `self.rows=len(mat); self.cols=len(mat[0]) if mat else 0`：记下行数列数。`len(mat[0]) if mat else 0` 是三元表达式：mat 非空取第一行的长度，空矩阵取 0，避免对空列表取 `mat[0]` 时报 IndexError。
  - 空矩阵时 rows=0、cols=0，p 是一张 1×1 的零表，任何 query 都会被下一行的范围检查拦下——防御链条是连贯的。
- `self.p=[[0]*(self.cols+1) for _ in range(self.rows+1)]`：建 (rows+1)×(cols+1) 的全零表。第 0 行、第 0 列保持 0 不动，就是"边界补零"；列表推导式一行生成二维结构。
- 双层循环填表：`self.p[i+1][j+1]=mat[i][j]+self.p[i][j+1]+self.p[i+1][j]-self.p[i][j]`。逐项读：当前元素 mat[i][j]，加上"不含本行、含到 j+1 列"的上前缀 p[i][j+1]，加上"含到本行、不含本列"的左前缀 p[i+1][j]，再减去被前两项重复计入的左上角 p[i][j]。写成一个表达式就是容斥公式本体。
  - 拿 2×2 矩阵的最后一格复查一遍：p[2][2] = mat[1][1] + p[1][2] + p[2][1] − p[1][1] = 4+3+4−1 = 10。若忘了减 p[1][1]，会得 11（左上角 1 被加了两次）；若把减号错写在别的项上，建表当场就不对。建表错误在后面随机对拍里一定会被抓出来。
  - 循环顺序 i 在外 j 在内还是反过来都行：递推只依赖"更上一行"和"更左一列"的值，两种顺序都满足依赖已就绪。
- `query` 里的 `if not (0<=r1<=r2<=self.rows and 0<=c1<=c2<=self.cols):`：链式比较加 and，要求行、列各自有序且都在表界内，否则抛 IndexError。这是防御式检查，r1>r2 或越界立刻暴露，不让错误悄悄传播。
- `p=self.p`：取个短别名少打几个 `self.`，纯书写习惯。
- `return p[r2][c2]-p[r1][c2]-p[r2][c1]+p[r1][c1]`：四项容斥查询——大块、减上条、减左条、加回左上角。
  - 用 2×2 的前缀表（[[0,0,0],[0,1,3],[0,4,10]]）算 query(0,1,2,2)：p[2][2]−p[0][2]−p[2][1]+p[0][1] = 10−0−4+0 = 6，对应右列 2+4。第 0 行的 0 在这里发挥了作用：贴顶边时"上条"整条是 0，公式不用变。

再看 `matrix_block_sum`，对应实现如下：

```python
def matrix_block_sum(mat, k):
    p = PrefixSum2D(mat)
    return [[p.query(max(0, i - k), max(0, j - k), min(p.rows, i + k + 1), min(p.cols, j + k + 1)) for j in range(p.cols)] for i in range(p.rows)]
```

- `p=PrefixSum2D(mat)`：先花 O(RC) 建一次前缀表，所有格子共用。
- `max(0,i-k)`：中心块的上边界。理想是 i−k，但负数说明越出顶边，钳回 0。列方向同理。
- `min(p.rows,i+k+1)`：下边界。理想右端是 i+k+1（半开，+1 是因为要包含第 i+k 行），超出总行数就钳回 rows，正好也是半开约定的合法右端。
- 双层列表推导：外层遍历行 i，内层遍历列 j，每个位置调一次 query，组装成答案矩阵。注意换行写法只是把长表达式折行，语义不变。
  - 钳制四件套 `max(0,…)`、`max(0,…)`、`min(rows,…)`、`min(cols,…)` 分别对应上、左、下、右四个方向，理想边界越界就被拉回；矩阵中间的格子（不越界）钳制前后值相同，所以中心格和边角格共用同一行代码。

## 5. 分段实现：按顺序运行

**本章接口：** `PrefixSum2D`、`matrix_block_sum(mat, k)`

### PrefixSum2D

In [1]:
class PrefixSum2D:

    def __init__(self, mat):
        self.rows = len(mat)
        self.cols = len(mat[0]) if mat else 0
        self.p = [[0] * (self.cols + 1) for _ in range(self.rows + 1)]
        for i in range(self.rows):
            for j in range(self.cols):
                self.p[i + 1][j + 1] = mat[i][j] + self.p[i][j + 1] + self.p[i + 1][j] - self.p[i][j]

    def query(self, r1, c1, r2, c2):
        if not (0 <= r1 <= r2 <= self.rows and 0 <= c1 <= c2 <= self.cols):
            raise IndexError('invalid rectangle')
        p = self.p
        return p[r2][c2] - p[r1][c2] - p[r2][c1] + p[r1][c1]

### matrix_block_sum

In [2]:
def matrix_block_sum(mat, k):
    p = PrefixSum2D(mat)
    return [[p.query(max(0, i - k), max(0, j - k), min(p.rows, i + k + 1), min(p.cols, j + k + 1)) for j in range(p.cols)] for i in range(p.rows)]

## 6. 运行示例与状态可视化

先预测结果再运行。手算表帮助解释过程；下面的表由当前实现计算。若表只展示输入输出，它并不代替内部状态轨迹。

In [3]:
from IPython.display import display, HTML
from html import escape

def show_table(headers, rows):
    """Render a small, explicit state trace; no network or hidden algorithm code."""
    head = ''.join('<th>' + escape(str(x)) + '</th>' for x in headers)
    body = ''.join('<tr>' + ''.join('<td>' + escape(str(x)) + '</td>' for x in row) + '</tr>' for row in rows)
    display(HTML('<table><thead><tr>' + head + '</tr></thead><tbody>' + body + '</tbody></table>'))

m=[[1,2],[3,4]]; p=PrefixSum2D(m)
show_table(['前缀表行','内容'],list(enumerate(p.p)))
show_table(['半开矩形','查询和'], [((r1,c1,r2,c2),p.query(r1,c1,r2,c2)) for r1,c1,r2,c2 in [(0,0,2,2),(1,1,2,2),(0,1,2,2)]])

前缀表行,内容
0,"[0, 0, 0]"
1,"[0, 1, 3]"
2,"[0, 4, 10]"


半开矩形,查询和
"(0, 0, 2, 2)",10
"(1, 1, 2, 2)",4
"(0, 1, 2, 2)",6


## 7. 正确性、适用条件与复杂度

用集合容斥看四个前缀：目标外的上方与左方各减一次，它们共同左上区域被减两次，再加回一次。

**代价：** 构建 O(RC) 时间与空间；查询 O(1)；整张block sum输出 O(RC)。

### 展开理解

**为什么对（建表）**：我们逐格验证递推式覆盖的面积。p[i][j+1] 覆盖行 [0,i)、列 [0,j+1)；p[i+1][j] 覆盖行 [0,i+1)、列 [0,j)。两者相加时，行 [0,i)、列 [0,j) 的左上区域被计了两次，减去 p[i][j]（恰好就是这块）把它修正为一次；再补上不在两者中的当前格 mat[i][j]，总和正好是行 [0,i+1)、列 [0,j+1) 的全部元素，不多不少。想验证这个面积论证，最快的办法还是拿 2×2 矩阵把四格全手算一遍——我们在第三节已经做过，四格全部严丝合缝。

**为什么对（查询）**：P[r2][c2] 是从原点出发的大矩形，包含目标矩形以及目标上方、左方两条多余区域。减 P[r1][c2] 去掉上方那条，减 P[r2][c1] 去掉左方那条；但这两条在目标矩形的左上方（行 [0,r1)、列 [0,c1) 那块）互相重叠，被减了两次，多减了一次，所以加回 P[r1][c1]。四项一加一减，剩下的恰好是目标矩形。

**复杂度**：建表跑满 rows×cols 个格子，每格常数次加减，O(RC) 时间和空间；每次查询固定四次取数三次加减，O(1)。具体数字：300×300 的矩阵是 9 万格，建表 9 万次操作，之后随便问多少次矩形和都是每问常数级；暴力做法每次查询最坏也要扫 9 万格，问一万次就是 9 亿次操作，差距一目了然。matrix_block_sum 输出整张答案矩阵，R×C 个格子每个 O(1)，总计还是 O(RC)。

一个帮助你记忆公式的口诀：查询是"**大 − 上 − 左 + 左上**"，建表是"**自己 + 上 + 左 − 左上**"。两者符号模式不同别背混——建表时你是在"从小拼大"（多加的重叠要减掉），查询时你是在"从大挖小"（多减的重叠要加回来）。忘掉哪个符号，最好的自检办法是拿 [[1,2],[3,4]] 手算一遍，四个格子十以内的加减，三十秒就能发现错。

## 8. 单元测试：每个用例在检查什么

“自动测试”部分 的断言逐条解释：

- `p=PrefixSum2D([[1,2],[3,4]])` 加 `assert p.query(0,0,2,2)==10 and p.query(1,1,2,2)==4`：正常值测试。整块和 10、右下单格 4，就是第三节手算演示的两个查询，同时验证了"贴边矩形"（从 (0,0) 开始）在补零边界下公式照样工作。
- `assert matrix_block_sum([[1,2],[3,4]],3)==[[10,10],[10,10]]`：正常值/边界钳制测试。k=3 的扩展范围远超 2×2 矩阵，四个中心的范围都被钳制成整块，答案全是 10。这条考察 max/min 钳制是否正确处理"四个方向同时越界"。
- 随机穷举段：`rng=Random(33)` 用固定随机种子保证可复现。对 1~4 行、1~4 列的全部尺寸，生成元素取 −3~3 的随机矩阵，然后**四层循环枚举所有合法半开矩形** (a,c,b,d)，用暴力 `sum(m[i][j] for i in range(a,b) for j in range(c,d))`（生成器直接扫矩形）对拍。几个细节值得注意：
  - 暴力里的行范围 `range(a,b)`、列范围 `range(c,d)` 天生半开，和 query 的约定一字不差地对齐。
  - 四层循环从 `a` 到 `b≥a`、`c` 到 `d≥c`，把空矩形（a==b 或 c==d，和为 0）和整块矩形（0,0,rows,cols）都包含在内。
  - 元素取 −3~3 含负数和零：前缀和不依赖元素正负，这条顺便证明了负元素场景公式不变。
  - 若查询公式的四个符号错了一个，任何一个非平凡矩形都会对不上，1×1 以上的矩阵立刻翻车——这就是"公式题靠穷举对拍"的底气。

In [4]:
p = PrefixSum2D([[1, 2], [3, 4]])

assert p.query(0, 0, 2, 2) == 10 and p.query(1, 1, 2, 2) == 4

assert matrix_block_sum([[1, 2], [3, 4]], 3) == [[10, 10], [10, 10]]

from random import Random

rng = Random(33)

for rows in range(1, 5):
    for cols in range(1, 5):
        m = [[rng.randrange(-3, 4) for _ in range(cols)] for _ in range(rows)]
        p = PrefixSum2D(m)
        for a in range(rows + 1):
            for b in range(a, rows + 1):
                for c in range(cols + 1):
                    for d in range(c, cols + 1):
                        assert p.query(a, c, b, d) == sum((m[i][j] for i in range(a, b) for j in range(c, d)))

print('N033: 所有本章断言通过')

N033: 所有本章断言通过


## 9. 练习：先尝试，再看提示

**练习 1：** 不存额外边界行列时比较代码复杂度。

**练习 2：** 扩展统计矩形内非零数。

<details>
<summary>展开思路提示（不是完整答案）</summary>

- **练习 1（不存额外边界行列时比较代码复杂度）**：提示：试着把前缀表开成 rows×cols（P[i][j] 直接对应原矩阵格子，不再补零）。你会发现建表和查询都得分类：i=0 或 j=0 时不能引用 p[i−1][j−1]（越界），要么在表达式里塞 `(i>0)` 的条件，要么循环前先手动填第一行第一列。建议各写一版，数一数分支和边界判断的条数，直观感受"补一圈零买来多少简洁"——这正是 notebook “思路推导”部分 说"避免为贴边区域写特殊分支"的含义。手算例子可继续用 [[1,2],[3,4]] 对拍两版结果：
  - 补零版查整块是 query(0,0,2,2)，四项公式直接出 10。
  - 不补零版查同一块时，"上方条"不存在，公式退化为 P[1][1]（矩阵右下格）加部分项，你得自己推一遍并单独处理，出错面明显变大。
- **练习 2（扩展统计矩形内非零数）**：提示：把"求和"换成"计数"——先构造 0/1 指示矩阵 `g[i][j] = 1 if mat[i][j]!=0 else 0`，对 g 建一套二维前缀和，查询公式一字不改，得到的就变成"矩形内非零格子的个数"。想通"前缀和技术对任何可加的统计量都成立"（和、计数、甚至布尔或），这一招就真正学到手了。验证可用含零的小矩阵如 [[0,1],[2,0]] 手算对拍：
  - 指示矩阵 g = [[0,1],[1,0]]，前缀表为 [[0,0,0],[0,0,1],[0,1,2]]。
  - 查询整块 (0,0,2,2) = 2，对应两个非零格（1 和 2 的位置）；查询左列 (0,0,2,1) = 1。
  - 同样套路还能数"矩形内负数个数"（指示条件改成 `<0`），你甚至不用改 PrefixSum2D 的任何一行。

</details>

## 10. 完整代码：可独立运行

下面这一格包含**全部导入、全部接口和验证用例**，不依赖前面的单元格。可以重启内核后只运行这一格，也可以复制到 `.py` 文件后运行。在 Notebook 中 Run All 时会再验证一次，这是为独立复制使用保留的完整版本。

In [5]:
# 完整实现：本单元格不依赖其他单元格。
class PrefixSum2D:

    def __init__(self, mat):
        self.rows = len(mat)
        self.cols = len(mat[0]) if mat else 0
        self.p = [[0] * (self.cols + 1) for _ in range(self.rows + 1)]
        for i in range(self.rows):
            for j in range(self.cols):
                self.p[i + 1][j + 1] = mat[i][j] + self.p[i][j + 1] + self.p[i + 1][j] - self.p[i][j]

    def query(self, r1, c1, r2, c2):
        if not (0 <= r1 <= r2 <= self.rows and 0 <= c1 <= c2 <= self.cols):
            raise IndexError('invalid rectangle')
        p = self.p
        return p[r2][c2] - p[r1][c2] - p[r2][c1] + p[r1][c1]

def matrix_block_sum(mat, k):
    p = PrefixSum2D(mat)
    return [[p.query(max(0, i - k), max(0, j - k), min(p.rows, i + k + 1), min(p.cols, j + k + 1)) for j in range(p.cols)] for i in range(p.rows)]

# 示例与回归测试
p = PrefixSum2D([[1, 2], [3, 4]])

assert p.query(0, 0, 2, 2) == 10 and p.query(1, 1, 2, 2) == 4

assert matrix_block_sum([[1, 2], [3, 4]], 3) == [[10, 10], [10, 10]]

from random import Random

rng = Random(33)

for rows in range(1, 5):
    for cols in range(1, 5):
        m = [[rng.randrange(-3, 4) for _ in range(cols)] for _ in range(rows)]
        p = PrefixSum2D(m)
        for a in range(rows + 1):
            for b in range(a, rows + 1):
                for c in range(cols + 1):
                    for d in range(c, cols + 1):
                        assert p.query(a, c, b, d) == sum((m[i][j] for i in range(a, b) for j in range(c, d)))

print('N033: 所有本章断言通过')

N033: 所有本章断言通过


## 11. 代表题与迁移

- **304. Range Sum Query 2D - Immutable**：这题就是 `PrefixSum2D` 的原题（不可变矩阵的区域和检索），练的是"补零边界 + 四项容斥"的二维前缀套路。该题矩阵可达 200×200、查询可达 10^4 次，二维前缀的"建一次表、次次 O(1)"正好压着数据规模的上限设计。
- **1314. Matrix Block Sum**：这题就是 `matrix_block_sum` 的原题（K 近邻块和），练的是"二维前缀查询 + max/min 钳制贴边范围"的组合应用。这题不给补零边界的话，四个角上的格子（i−k 或 j+k+1 越界）都要单独写分支，钳制写法能把所有格子统一成一次 query 调用。

**题面入口：** 

题目用于知识映射；本章完整代码遵循教学接口，不等于每道题的完整平台提交。

## 12. 掌握检查

合上代码，说明输入输出、核心变量的含义、每次更新为什么成立，以及一个边界情况。再完成一道练习，不能只凭“运行通过”判断掌握。

**继续学习：** [课程索引](../../COURSE_INDEX.md)。